# To Do: Whisper STT Fine-tuning

## 목표
AI Hub 외국인 한국어 발화 데이터를 사용해 `openai/whisper-large-v3-turbo`를 LoRA 방식으로 파인튜닝하고, test set에서 CER/WER을 확인한다.

## 순서

1. Colab 런타임을 GPU로 설정한다.
   - 가능하면 L4 또는 A100 사용
   - `!nvidia-smi`로 GPU 확인

2. 팀원 Google Drive 데이터 접근을 확인한다.
   - 팀원이 내 Colab 계정에 데이터 폴더 공유
   - 내 Drive에 바로가기 추가
   - `DATA_ROOT` 경로 수정

3. Google Drive를 마운트한다.
   - `drive.mount("/content/drive")`
   - `DATA_ROOT.exists()`가 `True`인지 확인

4. 데이터 구조를 확인한다.
   - wav 파일 개수 확인
   - json/csv 라벨 파일 개수 확인
   - 폴더 구조와 샘플 라벨 확인

5. manifest를 만든다.
   - wav 경로와 transcript를 매칭
   - 최종 컬럼은 `audio`, `sentence`
   - `Final manifest rows > 0`인지 확인

6. 데이터 품질을 확인한다.
   - 오디오 파일이 실제로 존재하는지 확인
   - 너무 짧거나 긴 오디오 제거
   - 문장 샘플을 직접 보고 전사문이 정상인지 확인

7. train / validation / test로 나눈다.
   - 기본은 train 90%, validation 5%, test 5%
   - manifest csv를 Drive에 저장

8. Hugging Face Dataset으로 변환한다.
   - `audio` 컬럼을 `Audio(sampling_rate=16000)`으로 변환
   - 샘플 하나를 출력해 오디오와 문장이 정상인지 확인

9. 먼저 `DEBUG_MODE = True`로 실행한다.
   - 작은 데이터셋으로 전체 파이프라인 테스트
   - 전처리, 모델 로드, LoRA 적용, 학습, 평가가 끝까지 되는지 확인

10. Whisper + LoRA 학습을 실행한다.
    - 모델: `openai/whisper-large-v3-turbo`
    - 방식: LoRA
    - batch size는 1부터 시작
    - gradient accumulation 사용
    - 처음에는 epoch 1로 시작

11. validation/test 평가를 확인한다.
    - CER 확인
    - WER 확인
    - 예측 문장과 정답 문장을 직접 비교

12. 전체 학습으로 확장한다.
    - `DEBUG_MODE = False`
    - 전체 데이터로 학습
    - 학습 완료 후 adapter와 processor 저장

## 핵심 체크포인트

- `DATA_ROOT.exists() == True`
- `wav files > 0`
- `json files > 0` 또는 `csv files > 0`
- `Final manifest rows > 0`
- 문장 샘플이 실제 한국어 전사문임
- `DEBUG_MODE=True`에서 끝까지 실행됨
- test set에서 CER/WER이 계산됨
- LoRA adapter가 저장됨


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB
GPU memory: 39.49 GB


In [2]:
!pip install -q \
  transformers \
  datasets \
  accelerate \
  evaluate \
  jiwer \
  peft \
  bitsandbytes \
  librosa \
  soundfile \
  wandb \
  scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 96.9 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import glob
import random
import shutil
import zipfile
import tarfile

import numpy as np
import pandas as pd
import torch
import soundfile as sf

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Union, Optional

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from google.colab import drive

from datasets import load_dataset, Dataset, DatasetDict, Audio
import evaluate

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

from huggingface_hub import login, notebook_login

In [4]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
MODEL_ID = "openai/whisper-large-v3-turbo"

LANGUAGE = "Korean"
TASK = "transcribe"

# 학습 산출물은 런타임이 끊겨도 남도록 내 Drive에 저장합니다.
OUTPUT_DIR = "/content/drive/MyDrive/ColabWork/whisper-large-v3-turbo-ko-lora"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("MODEL_ID:", MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)


MODEL_ID: openai/whisper-large-v3-turbo
OUTPUT_DIR: /content/drive/MyDrive/ColabWork/whisper-large-v3-turbo-ko-lora


In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
EXTRACT_DIR = Path("/content/dataset")
# 드라이브 안 외국인 발화자 데이터셋 zip 파일 위치
MY_DRIVE_ZIP = "/content/drive/MyDrive/외국인_발화.zip"

if not os.path.exists("/content/dataset"):
    print("🚀 내 구글 드라이브에서 파일을 복사해 오는 중...")

    # 드라이브에서 코랩 로컬로 파일 복사
    TEMP_ZIP = "/content/training.zip"
    !cp "{MY_DRIVE_ZIP}" {TEMP_ZIP}

    # 압축 해제
    if not EXTRACT_DIR.exists():
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    print("📦 압축 해제 중... (로컬 SSD를 사용하여 매우 빠름)")
    !unzip -q {TEMP_ZIP} -d {EXTRACT_DIR}

    if os.path.exists(TEMP_ZIP):
        os.remove(TEMP_ZIP)

    print("✅ 모든 준비가 끝났습니다!")

DATA_ROOT = EXTRACT_DIR

🚀 내 구글 드라이브에서 파일을 복사해 오는 중...
📦 압축 해제 중... (로컬 SSD를 사용하여 매우 빠름)
✅ 모든 준비가 끝났습니다!


In [8]:
from pathlib import Path

# 이제 드라이브 경로가 아닌 로컬 경로(/content/dataset)를 최상위로 설정합니다.
DATA_ROOT = Path("/content/dataset")

LABEL_DIR = DATA_ROOT / "label_data"
SOUND_DIR = DATA_ROOT / "sound_data"

print("🎯 [데이터 경로 점검 - 로컬 기준]")
print("현재 설정된 경로:", DATA_ROOT)
print("경로 존재 여부:", DATA_ROOT.exists())

🎯 [데이터 경로 점검 - 로컬 기준]
현재 설정된 경로: /content/dataset
경로 존재 여부: True


In [9]:
def show_tree(root: Path, max_depth: int = 3, max_items_per_dir: int = 10):
    root = Path(root)

    if not root.exists():
        print(f"Path does not exist: {root}")
        return

    root_depth = len(root.parts)

    for current_root, dirs, files in os.walk(root):
        current_path = Path(current_root)
        depth = len(current_path.parts) - root_depth

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth
        print(f"{indent}{current_path.name}/")

        for file in files[:max_items_per_dir]:
            print(f"{indent}  {file}")

        if len(files) > max_items_per_dir:
            print(f"{indent}  ... ({len(files) - max_items_per_dir} more files)")

show_tree(DATA_ROOT, max_depth=3, max_items_per_dir=10)

dataset/
  131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/
    01.데이터_new_20220719/
      1.Training/


In [10]:
def count_extensions(root: Path):
    counter = Counter()

    for path in root.rglob("*"):
        if path.is_file():
            counter[path.suffix.lower()] += 1

    return counter

ext_counter = count_extensions(DATA_ROOT)

print("File extension counts:")
for ext, count in ext_counter.most_common():
    print(f"{ext or '[no extension]'}: {count}")

File extension counts:
.wav: 43112
.csv: 43112
.json: 43112
.py: 1


In [11]:
wav_files = sorted(DATA_ROOT.rglob("*.wav"))
json_files = sorted(DATA_ROOT.rglob("*.json"))
csv_files = sorted(DATA_ROOT.rglob("*.csv"))

print("wav files:", len(wav_files))
print("json files:", len(json_files))
print("csv files:", len(csv_files))

print("\nSample wav files:")
for p in wav_files[:10]:
    print(p)

print("\nSample json files:")
for p in json_files[:10]:
    print(p)

print("\nSample csv files:")
for p in csv_files[:10]:
    print(p)

wav files: 43112
json files: 43112
csv files: 43112

Sample wav files:
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0017_20210828.wav
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0018_20210809.wav
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0030_20210802.wav
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0031_20210807.wav
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0034_20210731.wav
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0086_20210727.wav
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0112_20210805.wav
/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_

In [12]:
archive_files = []

for pattern in ["*.zip", "*.tar", "*.tar.gz", "*.tgz"]:
    archive_files.extend(DATA_ROOT.rglob(pattern))

print("Archive files:", len(archive_files))

for p in archive_files[:20]:
    print(p)

Archive files: 0


In [13]:
def preview_json(path: Path, max_chars: int = 4000):
    with open(path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)

    print("File:", path)
    print("Type:", type(data))

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys())[:50])
    elif isinstance(data, list):
        print("List length:", len(data))
        if len(data) > 0 and isinstance(data[0], dict):
            print("First item keys:", list(data[0].keys())[:50])

    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars])

if len(json_files) > 0:
    preview_json(json_files[0])
else:
    print("No JSON files found.")

File: /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/라벨링데이터/1. 한국일반/EX10QC226_EX0017_20210828.json
Type: <class 'dict'>
Top-level keys: ['fileName', 'file_info', 'transcription', 'SpeakerID', 'basic_info', 'residence_info', 'skill_info']
{
  "fileName": "EX10QC226_EX0017_20210828.wav",
  "file_info": {
    "speakerID": "EX0017",
    "sentenceID": "EX10QC226",
    "recordUnit": "android",
    "recordQuality": "16bit 16kHz MONO",
    "recordDate": "2021-08-28 22:42:04",
    "recordTime": "6.595"
  },
  "transcription": {
    "Reading": "",
    "ReadingLabelText": "",
    "Question": "당신은 어느 나라에서 왔나요? 한국에 온 이유는 무엇인가요?",
    "AnswerLabelText": "예 저는 우즈베키스탄에서 왔습니다 한국에 공부하러 왔습니다",
    "SentenceSpeechLV": "중"
  },
  "SpeakerID": "EX0017",
  "basic_info": {
    "gender": "M",
    "birthYear": "1984",
    "eduBackground": "석사이상"
  },
  "residence_info": {
    "country": "UZ",
    "residencePeriod": "5년 이상",
    "residenceCity": "KR-11"
  },
  "skill_info": {
 

In [14]:
import pandas as pd
from pathlib import Path

def preview_csv(path: Path, nrows: int = 5):
    df = pd.read_csv(path, encoding="cp949")
    print("File:", path)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head(nrows))
    return df

if len(csv_files) > 0:
    sample_csv_df = preview_csv(csv_files[0])
else:
    print("No CSV files found.")

File: /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/라벨링데이터/1. 한국일반/EX10QC226_EX0017_20210828.csv
Shape: (1, 25)
Columns: ['fileName', 'speakerID', 'sentenceID', 'recordUnit', 'recordQuality', 'recordDate', 'recordTime', 'Reading', 'ReadingLabelText', 'Question', 'AnswerLabelText', 'SentenceSpeechLV', 'SpeakerID', 'gender', 'birthYear', 'eduBackground', 'country', 'residencePeriod', 'residenceCity', 'languageClass', 'motherTongue', 'selfAssessment', 'topikGrade', 'LearningPeriod', 'learningSource']


,fileName,speakerID,sentenceID,recordUnit,recordQuality,recordDate,recordTime,Reading,ReadingLabelText,Question,...,eduBackground,country,residencePeriod,residenceCity,languageClass,motherTongue,selfAssessment,topikGrade,LearningPeriod,learningSource
0,EX10QC226_EX0017_20210828.wav,EX0017,EX10QC226,android,16bit 16kHz MONO,2021-08-28 22:42:04,6.595,NaN,NaN,당신은 어느 나라에서 왔나요? 한국에 온 이유는 무엇인가요?,...,석사이상,UZ,5년 이상,KR-11,기타,우즈베크어,상,6,232,학교 수업 (대학 포함)


In [15]:
def inspect_audio_files(paths, n=20):
    rows = []

    for path in tqdm(paths[:n]):
        try:
            info = sf.info(str(path))
            rows.append({
                "path": str(path),
                "samplerate": info.samplerate,
                "duration_sec": round(info.duration, 3),
                "channels": info.channels,
                "format": info.format,
                "subtype": info.subtype,
            })
        except Exception as e:
            rows.append({
                "path": str(path),
                "error": str(e),
            })

    return pd.DataFrame(rows)

audio_preview_df = inspect_audio_files(wav_files, n=30)
display(audio_preview_df)

  0%|          | 0/30 [00:00<?, ?it/s]

,path,samplerate,duration_sec,channels,format,subtype
0,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,6.595,1,WAV,PCM_16
1,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,15.140,1,WAV,PCM_16
2,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,9.996,1,WAV,PCM_16
3,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,11.689,1,WAV,PCM_16
4,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,13.463,1,WAV,PCM_16
5,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,9.960,1,WAV,PCM_16
6,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,11.040,1,WAV,PCM_16
7,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,10.160,2,WAV,PCM_16
8,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,13.812,1,WAV,PCM_16
9,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,16000,13.380,1,WAV,PCM_16


In [16]:
TEXT_KEY_CANDIDATES = [
    "text",
    "sentence",
    "transcript",
    "transcription",
    "utterance",
    "script",
    "label",
    "org_text",
    "normalized_text",
    "발화",
    "발화문",
    "전사",
    "전사문",
    "문장",
    "원문",
]

AUDIO_KEY_CANDIDATES = [
    "audio",
    "audio_path",
    "file",
    "file_path",
    "filename",
    "file_name",
    "wav",
    "wav_path",
    "path",
    "id",
    "audio_id",
    "음성파일",
    "파일명",
]


def find_first_value_by_keys(obj, key_candidates):
    """
    dict/list 중첩 구조에서 후보 key에 해당하는 첫 번째 값을 찾습니다.
    """
    if isinstance(obj, dict):
        for key in key_candidates:
            if key in obj and obj[key] not in [None, ""]:
                return obj[key]

        for value in obj.values():
            found = find_first_value_by_keys(value, key_candidates)
            if found not in [None, ""]:
                return found

    elif isinstance(obj, list):
        for item in obj:
            found = find_first_value_by_keys(item, key_candidates)
            if found not in [None, ""]:
                return found

    return None


def normalize_text(text: str) -> str:
    """
    Whisper 학습용 기본 텍스트 정규화.
    처음에는 과하게 정규화하지 않고 공백만 정리합니다.
    추후 데이터 샘플을 보고 숫자/기호/태그 처리 정책을 추가합니다.
    """
    if text is None:
        return ""

    text = str(text).strip()
    text = re.sub(r"\([^)]*\)", "", text) # () 안에 있는 설명 같은거 제거
    text = re.sub(r"\S+/\s*", "", text) # 문장에 덧붙인 말들 (어, )
    text = re.sub(r"\+", "", text) # 필요없는 특수기호 제거
    text = re.sub(r"\s+", " ", text)

    return text

In [17]:
def build_wav_index(wav_files):
    """
    파일명, stem 기준으로 wav 경로를 빠르게 찾기 위한 index.
    같은 파일명이 여러 개 있으면 리스트로 보관합니다.
    """
    index = defaultdict(list)

    for path in wav_files:
        path = Path(path)
        index[path.name].append(path)
        index[path.stem].append(path)

    return index


wav_index = build_wav_index(wav_files)

print("wav_index keys:", len(wav_index))

sample_keys = list(wav_index.keys())[:10]
for k in sample_keys:
    print(k, "->", wav_index[k][0])

wav_index keys: 86224
EX10QC226_EX0017_20210828.wav -> /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0017_20210828.wav
EX10QC226_EX0017_20210828 -> /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0017_20210828.wav
EX10QC226_EX0018_20210809.wav -> /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0018_20210809.wav
EX10QC226_EX0018_20210809 -> /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0018_20210809.wav
EX10QC226_EX0030_20210802.wav -> /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0030_20210802.wav
EX10QC226_EX0030_20210802 -> /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX10QC226_EX0030_20210802.wav
EX10QC226_EX0031_20210807.wav -> /co

In [18]:
def resolve_audio_path(audio_value, label_path: Path, wav_index):
    """
    라벨 안의 audio/file/path 값을 실제 wav 경로로 변환합니다.
    """
    candidates = []

    if audio_value is not None:
        audio_str = str(audio_value).strip()

        candidates.extend([
            audio_str,
            Path(audio_str).name,
            Path(audio_str).stem,
        ])

        # 확장자가 없는 경우 wav 확장자 후보 추가
        if not audio_str.lower().endswith(".wav"):
            candidates.append(audio_str + ".wav")
            candidates.append(Path(audio_str).name + ".wav")

    # JSON/CSV 파일명과 WAV 파일명이 같은 경우
    candidates.extend([
        label_path.with_suffix(".wav").name,
        label_path.stem,
        label_path.stem + ".wav",
    ])

    for cand in candidates:
        if cand in wav_index:
            return str(wav_index[cand][0])

    return None

In [19]:
def parse_json_label_file(json_path: Path, wav_index):
    try:
        with open(json_path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        # AI Hub 외국인 한국어 발화 데이터 구조에 맞게 추출
        audio_value = data.get("fileName")

        text = ""
        if "transcription" in data and "AnswerLabelText" in data["transcription"]:
            text = data["transcription"]["AnswerLabelText"]

        # 추출한 파일명으로 실제 wav 파일 경로 탐색
        audio_path = resolve_audio_path(audio_value, json_path, wav_index)
        text = normalize_text(text)

        if audio_path is None or text == "":
            return None

        return {
            "audio": audio_path,
            "sentence": text,
            "label_path": str(json_path),
            "source": "json",
        }

    except Exception as e:
        return {
            "audio": None,
            "sentence": "",
            "label_path": str(json_path),
            "source": "json",
            "error": str(e),
        }

json_records = []

for jp in tqdm(json_files):
    rec = parse_json_label_file(jp, wav_index)
    if rec is not None and rec.get("audio") is not None and rec.get("sentence", "") != "":
        json_records.append(rec)

json_manifest_df = pd.DataFrame(json_records)

print("JSON manifest rows:", len(json_manifest_df))

if len(json_manifest_df) > 0:
    display(json_manifest_df.head())
else:
    print("No valid JSON records parsed.")

  0%|          | 0/43112 [00:00<?, ?it/s]

JSON manifest rows: 9301


,audio,sentence,label_path,source
0,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,예 저는 우즈베키스탄에서 왔습니다 한국에 공부하러 왔습니다,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
1,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔어요 한국에 온 이유는 바로 한국의 좋은 교육 시스템입니다 ...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
2,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔습니다 한국에 대학원을 다니기 위해 왔습니다,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
3,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔어요 한국과 한국어 그리고 한국 문화에 대한 관심과 애정이 ...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
4,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,우즈베키스탄에서 왔는데 아제르바이잔 사람이에요 아제르바이잔에서 태어났고 자랐는데 우...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json


In [20]:
def guess_column(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}

    for cand in candidates:
        cand_lower = cand.lower()
        if cand_lower in lower_map:
            return lower_map[cand_lower]

    for c in columns:
        c_lower = str(c).lower()
        for cand in candidates:
            if cand.lower() in c_lower:
                return c

    return None


def parse_csv_label_file(csv_path: Path, wav_index):
    try:
        df = pd.read_csv(csv_path, encoding="CP949")

        records = []

        for _, row in df.iterrows():
            # RC 유형: ReadingLabelText 우선, QA 유형: AnswerLabelText 우선
            # 둘 다 있을 경우 비어있지 않은 것 사용
            text = ""

            answer = row.get("AnswerLabelText", None)
            reading = row.get("ReadingLabelText", None)

            if pd.notna(answer) and str(answer).strip():
                text = str(answer).strip()
            elif pd.notna(reading) and str(reading).strip():
                text = str(reading).strip()
            else:
                # 위 둘 다 없으면 기존 퍼지 매칭으로 폴백
                text_col = guess_column(df.columns, TEXT_KEY_CANDIDATES)
                if text_col and pd.notna(row[text_col]):
                    text = str(row[text_col]).strip()

            text = normalize_text(text)

            audio_value = row.get("fileName", None)
            audio_path = resolve_audio_path(audio_value, csv_path, wav_index)

            if audio_path is not None and text:
                records.append({
                    "audio": audio_path,
                    "sentence": text,
                    "label_path": str(csv_path),
                    "source": "csv",
                })

        return records

    except Exception as e:
        print("Failed:", csv_path, e)
        return []

csv_records = []

for cp in tqdm(csv_files):
    csv_records.extend(parse_csv_label_file(cp, wav_index))

csv_manifest_df = pd.DataFrame(csv_records)

print("CSV manifest rows:", len(csv_manifest_df))

if len(csv_manifest_df) > 0:
    display(csv_manifest_df.head())
    display(csv_manifest_df.sample(min(10, len(csv_manifest_df)), random_state=42))
else:
    print("No valid CSV records parsed.")

  0%|          | 0/43112 [00:00<?, ?it/s]

CSV manifest rows: 43112


,audio,sentence,label_path,source
0,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,예 저는 우즈베키스탄에서 왔습니다 한국에 공부하러 왔습니다,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
1,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔어요 한국에 온 이유는 바로 한국의 좋은 교육 시스템입니다 ...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
2,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔습니다 한국에 대학원을 다니기 위해 왔습니다,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
3,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔어요 한국과 한국어 그리고 한국 문화에 대한 관심과 애정이 ...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
4,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,우즈베키스탄에서 왔는데 아제르바이잔 사람이에요 아제르바이잔에서 태어났고 자랐는데 우...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv


,audio,sentence,label_path,source
18879,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,외국인들에게 소개하고 싶은 한국 음식은 김치를 비롯하여 비빔밥 불고기 잡채 같은 것...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
9183,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,오랜만에 다 같이 한강에 오니까 좋네요 저녁 먹을 시간이 다 되었는데 우리 뭐 시켜...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
37284,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,제가 오랫동안 꼭 해보고 싶은 일은 파라과이쟁이에요 오랫동안 하고 싶지만 시간이 없...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
23336,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,상추튀김은 양파와 고추 초절임을 상추에 싸 먹는 것이며 전라도 음식은 간이 세고 매...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
25638,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,침을 맞은 후에는 뜸 치료를 받았어요 간호사님이 허리 위에 세모나게 생긴 뜸을 올리...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
25645,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,침을 맞을 때와는 반대로 처음에는 아무 느낌이 들지 않았지만 점차 뜸을 놓은 부위가...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
38807,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,날씨가 더운 여름에 차가운 머드 속에서 시원하게 놀 수 있으니 열을 식히러 가기 좋...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
11277,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 한국에 온 후 주로 먹은 음식은 떡볶이였습니다 삼 년 전에 한국에 여행하러 왔...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
20814,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,물건을 살 때는 카드를 많이 사용합니다 왜냐면 카드를 사용하면 영수증도 있고 핸드폰...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv
22535,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,거리는 제 집에서 조금 멀었지만 통근버스가 있어서 출퇴근이 쉬웠습니다 식사까지 제공...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,csv


In [21]:
manifest_parts = []

if len(json_manifest_df) > 0:
    manifest_parts.append(json_manifest_df[["audio", "sentence", "label_path", "source"]])

if len(csv_manifest_df) > 0:
    manifest_parts.append(csv_manifest_df[["audio", "sentence", "label_path", "source"]])

if len(manifest_parts) == 0:
    raise ValueError(
        "No valid manifest records found. "
        "라벨 JSON/CSV 구조를 직접 확인한 뒤 TEXT_KEY_CANDIDATES/AUDIO_KEY_CANDIDATES를 수정해야 합니다."
    )

manifest_df = pd.concat(manifest_parts, ignore_index=True)

manifest_df = manifest_df.dropna(subset=["audio", "sentence"])
manifest_df = manifest_df[manifest_df["sentence"].str.len() > 0]

# 같은 audio가 JSON과 CSV 양쪽에서 잡힐 수 있으므로 중복 제거
manifest_df = manifest_df.drop_duplicates(subset=["audio"]).reset_index(drop=True)

print("Final manifest rows:", len(manifest_df))
print("Source counts:")
print(manifest_df["source"].value_counts())

display(manifest_df.head(20))

Final manifest rows: 43112
Source counts:
source
csv     33811
json     9301
Name: count, dtype: int64


,audio,sentence,label_path,source
0,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,예 저는 우즈베키스탄에서 왔습니다 한국에 공부하러 왔습니다,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
1,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔어요 한국에 온 이유는 바로 한국의 좋은 교육 시스템입니다 ...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
2,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔습니다 한국에 대학원을 다니기 위해 왔습니다,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
3,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔어요 한국과 한국어 그리고 한국 문화에 대한 관심과 애정이 ...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
4,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,우즈베키스탄에서 왔는데 아제르바이잔 사람이에요 아제르바이잔에서 태어났고 자랐는데 우...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
5,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔고요 한국의 방송 쪽에 관심이 많아서 한국 대학교에서 언론홍...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
6,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 카자흐스탄에서 왔어요 저는 한국에 온 이유는 유학이 때문이에요,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
7,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔고 공부하려고 왔습니다 지금 인하 대학교 대학원 다니고 있습니다,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
8,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 중앙아시아에 위치하는 우즈베키스탄이라는 나라에서 왔어요 한국에 대학교에서 공부...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json
9,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,저는 우즈베키스탄에서 왔습니다 고려인입니다 우리 조상들이 살았던 한국에 대해 더 많...,/content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성...,json


In [22]:
manifest_df["audio_exists"] = manifest_df["audio"].apply(lambda x: Path(x).exists())

print("Audio exists counts:")
print(manifest_df["audio_exists"].value_counts())

missing_df = manifest_df[~manifest_df["audio_exists"]]

if len(missing_df) > 0:
    print("Missing examples:")
    display(missing_df.head(10))

manifest_df = manifest_df[manifest_df["audio_exists"]].drop(columns=["audio_exists"]).reset_index(drop=True)

print("After removing missing audio:", len(manifest_df))

Audio exists counts:
audio_exists
True    43112
Name: count, dtype: int64
After removing missing audio: 43112


In [23]:
MAX_AUDIO_SECONDS = 30.0
MIN_AUDIO_SECONDS = 0.3

def get_duration(path):
    try:
        return sf.info(path).duration
    except Exception:
        return None

durations = []

for path in tqdm(manifest_df["audio"].tolist()):
    durations.append(get_duration(path))

manifest_df["duration_sec"] = durations

before = len(manifest_df)

manifest_df = manifest_df.dropna(subset=["duration_sec"])
manifest_df = manifest_df[
    (manifest_df["duration_sec"] >= MIN_AUDIO_SECONDS) &
    (manifest_df["duration_sec"] <= MAX_AUDIO_SECONDS)
].reset_index(drop=True)

after = len(manifest_df)

print("Before filtering:", before)
print("After filtering:", after)
print("Removed:", before - after)

display(manifest_df["duration_sec"].describe())

  0%|          | 0/43112 [00:00<?, ?it/s]

Before filtering: 43112
After filtering: 42834
Removed: 278


,duration_sec
count,42834.000000
mean,12.429867
std,3.356050
min,2.300562
25%,10.223594
50%,11.697500
75%,13.799750
max,29.999750


In [24]:
print("Random samples:")

for _, row in manifest_df.sample(min(30, len(manifest_df)), random_state=42).iterrows():
    print("-" * 100)
    print("audio:", row["audio"])
    print("duration:", row["duration_sec"])
    print("sentence:", row["sentence"])

Random samples:
----------------------------------------------------------------------------------------------------
audio: /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/1. 한국일반/EX14RC077_EX0095_20210921.wav
duration: 13.55975
sentence: 내가 뭘 샀는지 기억조차 안 내나 지금 입고 있는 치마도 처음 보는 것 같은데 그것도 새로 산 것 같다
----------------------------------------------------------------------------------------------------
audio: /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/2. 한국생활I/EX26RB145_EX0072_20210826.wav
duration: 12.71975
sentence: 국과 찌개가 발달한 우리나라 문화 덕에 국물을 활용한 요리를 배우는 것은 색다른 즐거움을 줄 수 있다
----------------------------------------------------------------------------------------------------
audio: /content/dataset/131.인공지능_학습을_위한_외국인_한국어_발화_음성_데이터/01.데이터_new_20220719/1.Training/원천데이터/2. 한국생활I/EX22RB044_EX0112_20210730.wav
duration: 10.19975
sentence: 지하철을 타고 집으로 돌아오는 길에 건강 검진을 꼭 한번 받아 봐야겠다는 생각이 들었습니다
---------------------------

In [25]:
MIN_TEXT_LEN = 1
MAX_TEXT_LEN = 300

before = len(manifest_df)

manifest_df["text_len"] = manifest_df["sentence"].str.len()

manifest_df = manifest_df[
    (manifest_df["text_len"] >= MIN_TEXT_LEN) &
    (manifest_df["text_len"] <= MAX_TEXT_LEN)
].reset_index(drop=True)

after = len(manifest_df)

print("Before text filtering:", before)
print("After text filtering:", after)
print("Removed:", before - after)

display(manifest_df["text_len"].describe())

Before text filtering: 42834
After text filtering: 42834
Removed: 0


,text_len
count,42834.000000
mean,59.982584
std,14.973335
min,8.000000
25%,52.000000
50%,56.000000
75%,63.000000
max,202.000000


In [26]:
train_df, temp_df = train_test_split(
    manifest_df,
    test_size=0.10,
    random_state=42,
    shuffle=True,
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

train: 38550
valid: 2142
test: 2142


In [27]:
MANIFEST_DIR = Path("/content/drive/MyDrive/ColabWork/AIHub505_manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

train_path = MANIFEST_DIR / "train.csv"
valid_path = MANIFEST_DIR / "valid.csv"
test_path = MANIFEST_DIR / "test.csv"
all_path = MANIFEST_DIR / "all_manifest.csv"

manifest_df.to_csv(all_path, index=False, encoding="utf-8-sig")
train_df.to_csv(train_path, index=False, encoding="utf-8-sig")
valid_df.to_csv(valid_path, index=False, encoding="utf-8-sig")
test_df.to_csv(test_path, index=False, encoding="utf-8-sig")

print("Saved manifest files:")
print(all_path)
print(train_path)
print(valid_path)
print(test_path)

Saved manifest files:
/content/drive/MyDrive/ColabWork/AIHub505_manifests/all_manifest.csv
/content/drive/MyDrive/ColabWork/AIHub505_manifests/train.csv
/content/drive/MyDrive/ColabWork/AIHub505_manifests/valid.csv
/content/drive/MyDrive/ColabWork/AIHub505_manifests/test.csv


In [28]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["audio", "sentence"]].reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df[["audio", "sentence"]].reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df[["audio", "sentence"]].reset_index(drop=True)),
})

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 38550
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2142
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2142
    })
})

In [29]:
from datasets import Audio

# decode=True 옵션을 명시적으로 전달
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000, decode=True))

# 데이터를 가져올 때 바로 딕셔너리로 변환되었는지 확인
sample = dataset["train"][0]

print("Sentence:", sample["sentence"])
print("Audio Type:", type(sample["audio"])) # <class 'dict'> 여야 합니다.
if isinstance(sample["audio"], dict):
    print("Sampling rate:", sample["audio"]["sampling_rate"])

Sentence: 해외여행 중 만약 여권을 잃어버렸다면 첫 번째로 본인의 대사관에 연락주셔야 한다고 생각합니다
Audio Type: <class 'datasets.features._torchcodec.AudioDecoder'>


In [30]:
DEBUG_MODE = False

if DEBUG_MODE:
    train_n = min(100, len(dataset["train"]))
    valid_n = min(20, len(dataset["validation"]))
    test_n = min(20, len(dataset["test"]))

    working_dataset = DatasetDict({
        "train": dataset["train"].select(range(train_n)),
        "validation": dataset["validation"].select(range(valid_n)),
        "test": dataset["test"].select(range(test_n)),
    })
else:
    working_dataset = dataset

working_dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 38550
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2142
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2142
    })
})

# Whisper large-v3-turbo LoRA 학습

여기부터는 `working_dataset`이 정상적으로 만들어졌다는 전제에서 실행합니다.  
처음에는 `DEBUG_MODE=True` 상태로 작은 데이터가 돌아가는지 확인하고, 이후 `DEBUG_MODE=False`로 전체 학습을 실행하세요.

In [31]:
# 학습 관련 하이퍼파라미터
# Colab L4/A100 기준으로 시작하기 좋은 보수적 설정입니다.
# T4에서는 MODEL_ID를 small/medium으로 바꿔 파이프라인 검증을 먼저 권장합니다.

USE_8BIT = False          # 메모리가 부족하면 True로 변경해 볼 수 있습니다.
USE_FP16 = False           # Colab GPU에서는 보통 True
USE_BF16 = True          # A100이고 bf16을 쓰고 싶으면 True로 변경

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
#LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj"]

PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2

LEARNING_RATE = 5e-5
NUM_TRAIN_EPOCHS = 2

# DEBUG_MODE=True일 때는 자주 평가/저장해도 부담이 적습니다.
EVAL_STEPS = 500
SAVE_STEPS = 500
LOGGING_STEPS = 50

GENERATION_MAX_LENGTH = 225

print("USE_8BIT:", USE_8BIT)
print("LoRA:", {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT})
print("Effective batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

USE_8BIT: False
LoRA: {'r': 16, 'alpha': 16, 'dropout': 0.05}
Effective batch size: 16


In [32]:
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language=LANGUAGE,
    task=TASK,
)

print(processor)

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

WhisperProcessor:
- feature_extractor: WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 128,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}

- tokenizer: WhisperTokenizer(name_or_path='openai/whisper-large-v3-turbo', vocab_size=50257, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50257: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50258: AddedToken("<|startoftranscript|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50259: AddedToken("<|en|>", 

In [33]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # audio["array"]는 datasets.Audio(sampling_rate=16000) 덕분에 16kHz로 로드됩니다.
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids

    return batch

os.makedirs("/content/cache", exist_ok=True)

# 혹시 모를 cache 삭제 (이거 안하면 이전 cache로 training함.)
if os.path.exists("/content/cache"):
    shutil.rmtree("/content/cache")
os.makedirs("/content/cache", exist_ok=True)

# audio/sentence 컬럼을 log-Mel feature와 토큰 ID로 변환 후 캐시에 저장합니다.
# 이렇게 해야지 training 속도가 빨라
vectorized_dataset = working_dataset.map(
    prepare_dataset,
    remove_columns=working_dataset["train"].column_names,
    desc="Extracting log-Mel features and tokenizing labels",
    cache_file_names={
        "train": "/content/cache/train.arrow",
        "validation": "/content/cache/valid.arrow",
        "test": "/content/cache/test.arrow",
    }
)

vectorized_dataset

Extracting log-Mel features and tokenizing labels:   0%|          | 0/38550 [00:00<?, ? examples/s]

Extracting log-Mel features and tokenizing labels:   0%|          | 0/2142 [00:00<?, ? examples/s]

Extracting log-Mel features and tokenizing labels:   0%|          | 0/2142 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 38550
    })
    validation: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 2142
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 2142
    })
})

In [34]:
# 캐시에 잘 들어갔는지 확인
print(vectorized_dataset["train"].cache_files)
print("train:", len(vectorized_dataset["train"]))
print("validation:", len(vectorized_dataset["validation"]))
print("test:", len(vectorized_dataset["test"]))

[{'filename': '/content/cache/train.arrow'}]
train: 38550
validation: 2142
test: 2142


In [35]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        batch["input_features"] = batch["input_features"].to(torch.bfloat16)

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        # BOS 토큰이 전체 라벨 앞에 붙어 있으면 제거합니다.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.bos_token_id,
)

In [36]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def normalize_for_eval(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # -100은 loss 계산에서 무시되는 값이므로 decode 전에 pad_token_id로 복구합니다.
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True,
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True,
    )

    pred_str = [normalize_for_eval(s) for s in pred_str]
    label_str = [normalize_for_eval(s) for s in label_str]

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {
        "wer": wer,
        "cer": cer,
    }

In [37]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [38]:
if USE_8BIT:
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
    )

    model = prepare_model_for_kbit_training(model)
else:
    if USE_BF16:
        target_dtype = torch.bfloat16
    elif USE_FP16:
        target_dtype = torch.float16
    else:
        target_dtype = torch.float32

    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=target_dtype,
        device_map="auto",
    )

# 학습 중에는 cache를 끄는 것이 gradient checkpointing과 충돌을 피하는 데 안전합니다.
model.config.use_cache = False
model.generation_config.use_cache = True

# 한국어 전사 태스크를 명시합니다.
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 6,553,600 || all params: 815,431,680 || trainable%: 0.8037


In [39]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,

    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=False,

    predict_with_generate=True,
    generation_max_length=GENERATION_MAX_LENGTH,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,

    logging_steps=LOGGING_STEPS,
    report_to="none",  # wandb를 쓰려면 "wandb"로 변경하세요.

    remove_unused_columns=False,
    label_names=["labels"],

    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    push_to_hub=False,
    dataloader_num_workers=0,
    group_by_length=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=vectorized_dataset["train"],
    eval_dataset=vectorized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer

In [40]:
# 1. 학습 전 베이스라인 모델 성능 평가 (Ablation Study)
print("=== 베이스라인 평가 ===")
with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    baseline_metrics = trainer.evaluate(
        eval_dataset=vectorized_dataset["test"],
        metric_key_prefix="baseline_test",
    )
print(baseline_metrics)

# 베이스라인 결과 Drive에 저장 (런타임 죽어도 보존)
import json
with open("/content/drive/MyDrive/ColabWork/baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f)

=== 베이스라인 평가 ===


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

{'baseline_test_loss': 1.2674353122711182, 'baseline_test_model_preparation_time': 0.0203, 'baseline_test_wer': 0.2857184630680157, 'baseline_test_cer': 0.100147194688464, 'baseline_test_runtime': 708.0941, 'baseline_test_samples_per_second': 3.025, 'baseline_test_steps_per_second': 0.378}


In [41]:
# 2. 파인튜닝 학습 시작
print("\n=== LoRA 파인튜닝 시작 ===")
train_result = trainer.train()

print(train_result)


=== LoRA 파인튜닝 시작 ===


Step,Training Loss,Validation Loss,Model Preparation Time,Wer,Cer
500,0.300965,0.145566,0.020300,0.122361,0.040191
1000,0.268591,0.130180,0.020300,0.101397,0.032615
1500,0.244281,0.120258,0.020300,0.093179,0.029906
2000,0.240417,0.114176,0.020300,0.088533,0.028641
2500,0.213843,0.109222,0.020300,0.082697,0.026685
3000,0.211539,0.106065,0.020300,0.077470,0.025063
3500,0.206277,0.103908,0.020300,0.077122,0.024923
4000,0.206042,0.101989,0.020300,0.074857,0.024085
4500,0.201694,0.100943,0.020300,0.074828,0.024131


TrainOutput(global_step=4820, training_loss=0.25359777647429976, metrics={'train_runtime': 27306.2527, 'train_samples_per_second': 2.824, 'train_steps_per_second': 0.177, 'total_flos': 1.32615904886784e+20, 'train_loss': 0.25359777647429976, 'epoch': 2.0})


In [46]:
# 학습 로그 저장 (스텝별 loss, CER, WER)
import pandas as pd
log_df = pd.DataFrame(trainer.state.log_history)
log_df.to_csv("/content/drive/MyDrive/ColabWork/train_log.csv", index=False)
print("학습 로그 저장 완료")

학습 로그 저장 완료


In [42]:
# 학습된 LoRA adapter와 processor 저장
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)

Saved to: /content/drive/MyDrive/ColabWork/whisper-large-v3-turbo-ko-lora


# Validation / Test 평가

먼저 validation set으로 확인한 뒤, 최종적으로 test set을 평가합니다.  
`DEBUG_MODE=True`일 때는 샘플 수가 적으므로 점수 자체보다 코드가 끝까지 도는지 확인하세요.

In [43]:
with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    validation_metrics = trainer.evaluate(
        eval_dataset=vectorized_dataset["validation"],
        metric_key_prefix="validation",
    )

print(validation_metrics)

{'validation_loss': 0.10198938101530075, 'validation_model_preparation_time': 0.0203, 'validation_wer': 0.07523447254566044, 'validation_cer': 0.024154739358564378, 'validation_runtime': 703.3771, 'validation_samples_per_second': 3.045, 'validation_steps_per_second': 0.381, 'epoch': 2.0}


In [44]:
# 학습된 모델로 Test Set 최종 평가
print("=== 학습 후(Trained) Test Set 평가 ===")
with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    test_metrics = trainer.evaluate(
        eval_dataset=vectorized_dataset["test"],
        metric_key_prefix="test",
    )
print("Trained Metrics:", test_metrics)

# Ablation Study 결과 비교 출력
print("\n" + "="*50)
print("📊 Ablation Study 결과 (성능 향상도 비교)")
print("="*50)
print(f"[학습 전 Base] CER: {baseline_metrics['baseline_test_cer']:.4f} | WER: {baseline_metrics['baseline_test_wer']:.4f}")
print(f"[학습 후 LoRA] CER: {test_metrics['test_cer']:.4f} | WER: {test_metrics['test_wer']:.4f}")
print("="*50)
# (참고: CER과 WER은 낮을수록 성능이 좋음을 의미합니다.)

=== 학습 후(Trained) Test Set 평가 ===


Trained Metrics: {'test_loss': 0.10187262296676636, 'test_model_preparation_time': 0.0203, 'test_wer': 0.07205099713433534, 'test_cer': 0.023637274706002097, 'test_runtime': 703.393, 'test_samples_per_second': 3.045, 'test_steps_per_second': 0.381, 'epoch': 2.0}

📊 Ablation Study 결과 (성능 향상도 비교)
[학습 전 Base] CER: 0.1001 | WER: 0.2857
[학습 후 LoRA] CER: 0.0236 | WER: 0.0721


In [45]:
# 샘플 예측 확인
# 점수만 보지 말고 실제로 어떤 문장을 어떻게 인식하는지도 반드시 확인하세요.

def transcribe_sample(dataset_split, index=0):
    item = dataset_split[index]
    input_features = torch.tensor(item["input_features"]).unsqueeze(0).to(model.device).to(torch.bfloat16)

    with torch.no_grad():
        generated_ids = model.generate(
            input_features=input_features,
            max_length=GENERATION_MAX_LENGTH,
        )

    pred_text = processor.tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0]

    label_ids = item["labels"]
    ref_text = processor.tokenizer.decode(
        label_ids,
        skip_special_tokens=True,
    )

    print("REFERENCE:", normalize_for_eval(ref_text))
    print("PREDICTED :", normalize_for_eval(pred_text))


with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for i in range(min(5, len(vectorized_dataset["test"]))):
        print("=" * 100)
        transcribe_sample(vectorized_dataset["test"], index=i)

REFERENCE: 한 일 주일 전부터 얼굴에 자꾸 뭐가 나는데 깨끗하게 세수를 해도 좋아지지 않네요 약을 좀 바를까요
PREDICTED : 한 일주일 전부터 얼굴에 자꾸 뭐가 나는데 깨끗하게 세수를 해도 좋아지지 않네요 약을 좀 바를까요
REFERENCE: 사람들의 다양한 성격과 일 처리 능력에 대한 분석이 나날이 늘면서 그에 관한 유형 검사도 인기다
PREDICTED : 사람들의 다양한 성격과 일절의 능력에 대한 분석이 나날이 늘면서 그에 관한 유형 검사도 인기다
REFERENCE: 네 방으로 예약하신 분들은 미리 주문을 해 주셔야 하고요 술은 한 병 이상 주문해 주셔야 합니다
PREDICTED : 네 방으로 약하신 분들은 미리 주문을 해 주셔야 하고요 술은 한 병 이상 주문해 주셔야 합니다
REFERENCE: 신청하셨다니 다행이에요 신청하신 내역은 오 분 안에 반영되니 정상적으로 수강 신청이 되었는지 한 번 더 확인해 주시면 될 것 같습니다
PREDICTED : 신청하셨다니 다행이에요 신청하신 내역은 오 분 안에 반영되니 정상적으로 수강 신청이 되었는지 한 번 더 확인해 주시면 될 것 같습니다
REFERENCE: 동양사상이 바탕이 되어 한국에서 주로 여성들이 육아를 책임지는 것은 출산율이 줄어드는 원인이 된다
PREDICTED : 동양 사상이 바탕이 되어 한국에서 주로 여성들이 육아를 책임지는 것은 출산율이 줄어드는 원인이 된다


In [47]:
with open("/content/drive/MyDrive/ColabWork/validation_metrics.json", "w") as f:
    json.dump(validation_metrics, f)

In [48]:
with open("/content/drive/MyDrive/ColabWork/final_metrics.json", "w") as f:
    json.dump({
        "baseline": baseline_metrics,
        "validation": validation_metrics,
        "test": test_metrics,
    }, f, indent=2)
print("최종 결과 저장 완료")

최종 결과 저장 완료


# 전체 데이터 학습으로 전환하는 방법

위 과정이 정상적으로 끝나면 전처리 파트의 `DEBUG_MODE = True`를 `False`로 바꾸고 런타임을 다시 실행하세요.  
전체 데이터 학습 시에는 Colab 런타임이 끊길 수 있으므로 `OUTPUT_DIR`이 반드시 Google Drive 안에 있는지 확인해야 합니다.